# Heston Stochastic-Volatility Conditioning

This notebook reproduces the Heston row of Figure 4 in *LatentFlow: A General Framework for Conditioning Stochastic Processes* (Sharrock, Astfalck & Moss, 2026). A **stochastic-volatility process** is written as the deterministic image of Gaussian innovations, pushed through an Euler–Maruyama simulator of the log-price $X_t$ and instantaneous variance $V_t$:

$$dX_t = \left(\mu - \tfrac{1}{2} V_t\right)dt + \sqrt{V_t}\,dW^X_t, \qquad dV_t = \kappa(\theta - V_t)\,dt + \xi\sqrt{V_t}\,dW^V_t,$$

with leverage correlation $\mathrm{corr}(dW^X_t, dW^V_t) = \rho < 0$. The price index is $100\,e^{X_t}$. Guidance then conditions the paths on

1. **data**: noisy observations of the price over the early window $t \le 0.2$;
2. **an event**: a market shock, where the price falls below $87$ at some point and volatility spikes above $0.42$;
3. **both** at once.

Volatility is never observed directly, so conditioning on prices alone must infer it.

## 1. Imports

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from flow_inference.conditions import BinaryCondition, EqualityCondition, ProductCondition
from flow_inference.flows import GaussianIncrementFlow
from flow_inference.guidance import GuidanceMC
from flow_inference.maps import BaseMap
from flow_inference.sampler import IntegratorSampler

sys.path.insert(0, str(Path.cwd().parent))  # make examples/plotting.py importable
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnchoredOffsetbox, TextArea, VPacker
from plotting import as_numpy, plot_traces

torch.set_default_dtype(torch.float64)

## 2. Configuration

One trading year discretised into $N = 126$ steps; the grid is defined at module level so the map and the conditions can both read it. The second cell holds the plotting and event-rate helpers reused by every figure. Price and volatility live on different scales, so each panel puts the price index on a twin axis, as in the paper.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
torch.manual_seed(SEED)  # reproducible runs

num_samples = 96  # paths per prior / guided batch
NUM_PLOTTED = 40  # paths drawn per panel; the rough Heston paths clutter beyond this
num_times = 126  # trading days in one year
horizon = 1.0  # time horizon in years
times = torch.linspace(0.0, horizon, num_times, device=device)
NUM_STEPS = 100  # flow integration steps

OBS_WINDOW = 0.2  # observations are taken on t <= OBS_WINDOW
OBS_NOISE = 0.005  # observation noise standard deviation on the log-price (~0.5%)
MIN_PRICE, MAX_VOL = 87.0, 0.42  # the market-shock thresholds

CHANNELS = ["price index", "volatility"]
COLORS = {"price index": "#2563eb", "volatility": "#dc2626"}
EVENT_LINES = [  # components of the event, for the threshold lines and the event box
    {"text": f"min price < {MIN_PRICE:g}", "color": COLORS["price index"], "channel": "price index", "y": MIN_PRICE, "span": "path"},
    {"text": f"max vol > {MAX_VOL}", "color": COLORS["volatility"], "channel": "volatility", "y": MAX_VOL, "span": "path"},
]

In [ ]:
def price_index(samples):
    """(B, 2, num_times) samples -> (B, num_times) price index 100 exp(X_t)."""
    return 100.0 * torch.exp(samples[:, 0, :])


def plot_paths(ax, samples, title, labels=True):
    """The first NUM_PLOTTED samples' volatility on `ax` (left) and price index on a twin axis
    (right), as in the paper. Returns the twin."""
    price_ax = ax.twinx()
    ax.set_zorder(price_ax.get_zorder() + 1)  # volatility, legend and event box above the prices
    ax.patch.set_visible(False)
    count = min(samples.shape[0], NUM_PLOTTED)
    plot_traces(samples[:, 1], x=times, color=COLORS["volatility"], count=count, alpha=0.45, linewidth=1.1, ax=ax)
    plot_traces(price_index(samples), x=times, color=COLORS["price index"], count=count, alpha=0.45, linewidth=1.1,
                ax=price_ax)
    ax.set_title(title, pad=8)
    ax.set_xlabel("time")
    ax.set_ylim(0.0, 0.75)
    price_ax.set_ylim(65, 125)
    if labels:
        ax.set_ylabel("volatility", color=COLORS["volatility"])
        price_ax.set_ylabel("price index", color=COLORS["price index"])
    ax.grid(True, color="#e5e7eb", linewidth=0.8)
    ax.margins(x=0.01)
    return price_ax


def channel_legend(ax, loc):
    handles = [Line2D([], [], color=COLORS[name], linewidth=2.0, label=name) for name in CHANNELS]
    ax.legend(handles=handles, loc=loc)


def show_event(ax, price_ax, loc):
    threshold_lines({"price index": price_ax, "volatility": ax})
    event_box(ax, loc)


def plot_observations(ax, observations, channels, mask, transform=lambda y: y):
    """Observed values as black dots."""
    for channel in channels:
        ax.scatter(as_numpy(times[mask]), as_numpy(transform(observations[channel, mask])),
                   color="black", edgecolor="white", linewidth=0.6, s=30, zorder=5)


def event_box(ax, loc):
    """The event's components as coloured lines of text in a white box, as in the paper."""
    lines = [TextArea(line["text"], textprops=dict(color=line["color"], fontsize=9)) for line in EVENT_LINES]
    box = AnchoredOffsetbox(loc=loc, child=VPacker(children=lines, align="left", pad=0, sep=2),
                            pad=0.4, borderpad=0.6, frameon=True)
    box.patch.set_boxstyle("round")
    box.patch.set_edgecolor("#9ca3af")
    ax.add_artist(box)


def threshold_lines(axes_by_channel):
    """A dashed line at every event threshold: across the whole path for extremes, over the
    final stretch for terminal conditions."""
    for line in EVENT_LINES:
        xmin = 0.0 if line["span"] == "path" else 0.85
        axes_by_channel[line["channel"]].axhline(line["y"], xmin=xmin, color=line["color"],
                                                 linestyle="--", linewidth=1.5, zorder=4)


def smooth_max(values: torch.Tensor, tau: float) -> torch.Tensor:
    """Log-sum-exp maximum over time: differentiable, and within tau * log(num_times) of the hard max."""
    return tau * torch.logsumexp(values / tau, dim=-1)


def smooth_min(values: torch.Tensor, tau: float) -> torch.Tensor:
    return -smooth_max(-values, tau)


def event_rate(condition: BinaryCondition, samples: torch.Tensor) -> float:
    """Fraction of samples meeting every component of the event (all margins > 0)."""
    margins = condition.smooth_eval_condition(samples).squeeze(1)  # (B, components)
    return (margins.min(dim=-1).values > 0).double().mean().item()

## 3. The Heston SDE

`HestonMap` is the generator $T_\vartheta$: it integrates the correlated two-factor Heston model with Euler–Maruyama. Log-price and volatility are coupled channels of one asset, so the map consumes the whole `(B, 2, num_times)` latent tensor: the first time slice perturbs the initial state and the rest supply the Brownian increments, correlated through $\rho$. The variance mean-reverts to $\theta$ at speed $\kappa$ with volatility-of-volatility $\xi$; clamps and a `tanh` soft-clamp on the log-price keep the paths bounded.

In [ ]:
class HestonMap(BaseMap):
    """Coupled (log-price, volatility) Euler-Maruyama simulator for the Heston model."""

    def __init__(
        self,
        num_times: int,
        mu: float = 0.05,
        kappa: float = 3.0,
        theta: float = 0.04,
        xi: float = 0.55,
        rho: float = -0.65,
    ):
        super().__init__(
            latent_dim=num_times,
            output_shape=(num_times,),
            channels=2,
            device=device,
            dtype=torch.float64,
        )
        self.num_times = num_times
        self.dt = horizon / (num_times - 1)
        self.sqrt_dt = self.dt ** 0.5
        self.mu = mu
        self.kappa = kappa
        self.theta = theta
        self.xi = xi
        self.rho = rho

    def _map(self, z: torch.Tensor) -> torch.Tensor:
        """z: (B, 2, num_times) iid N(0, 1) -> (B, 2, num_times) (log-price, volatility) paths."""
        B, C, N = z.shape
        if C != 2 or N != self.num_times:
            raise ValueError(f"expected latent shape (B, 2, {self.num_times}), got {tuple(z.shape)}")

        log_price = 0.015 * z[:, 0, 0]
        variance = (0.04 + 0.010 * z[:, 1, 0]).clamp(0.006, 0.25)
        log_price_path = [log_price]
        volatility_path = [variance.sqrt()]

        for k in range(self.num_times - 1):
            variance_pos = variance.clamp_min(1e-5)
            dW_variance = self.sqrt_dt * z[:, 1, k + 1]
            dW_price = self.rho * dW_variance + (1.0 - self.rho**2) ** 0.5 * self.sqrt_dt * z[:, 0, k + 1]

            log_price = log_price + (self.mu - 0.5 * variance_pos) * self.dt + variance_pos.sqrt() * dW_price
            variance = variance + self.kappa * (self.theta - variance_pos) * self.dt + self.xi * variance_pos.sqrt() * dW_variance
            variance = variance.clamp(1e-5, 0.80)
            log_price = 0.75 * torch.tanh(log_price / 0.75)

            log_price_path.append(log_price)
            volatility_path.append(variance.sqrt())

        return torch.stack(
            [torch.stack(log_price_path, dim=-1), torch.stack(volatility_path, dim=-1)],
            dim=1,
        )  # (B, 2, num_times)

## 4. Prior

`GaussianIncrementFlow` runs the flow over the iid Gaussian increments and pushes them through the map, so sampling it without guidance gives unconditioned prior paths.

In [ ]:
flow = GaussianIncrementFlow(map=HestonMap(num_times), device=device)
sampler = IntegratorSampler(flow=flow)
prior_samples = sampler.sample(num_samples, num_steps=NUM_STEPS).detach()

fig, ax = plt.subplots(figsize=(6, 3.6))
plot_paths(ax, prior_samples, "Prior")
channel_legend(ax, loc="upper left")
plt.tight_layout()

## 5. Conditioned on Data

Take one prior path as the "truth" and observe only its price at every third trading day of the early window $t \le 0.2$, with log-price noise of standard deviation $0.005$. `NoisyObservation` is an `EqualityCondition` on the log-price channel whose `sigma` is that noise level. The observations fix the early price; the volatility they imply is inferred, not observed.

In [ ]:
truth = prior_samples[0]
obs_mask = (times <= OBS_WINDOW) & (torch.arange(num_times, device=device) % 3 == 0)
obs_channels = torch.tensor([0], device=device)  # log-price only
generator = torch.Generator(device=device).manual_seed(SEED)
observations = truth + OBS_NOISE * torch.randn(truth.shape, device=device, generator=generator)


class NoisyObservation(EqualityCondition):
    """Gaussian observations of `channels` at the masked times, with noise standard deviation `sigma`."""

    def __init__(self, observations: torch.Tensor, channels: torch.Tensor, mask: torch.Tensor, sigma: float):
        super().__init__(sigma=sigma)
        self.register_buffer("observations", observations)
        self.register_buffer("channels", channels)
        self.register_buffer("mask", mask)

    def residual(self, x: torch.Tensor) -> torch.Tensor:
        selected = x[:, self.channels][:, :, self.mask]
        target = self.observations[self.channels][:, self.mask]
        return selected - target[None, :, :]


data_condition = NoisyObservation(observations, obs_channels, obs_mask, sigma=OBS_NOISE)
data_guidance = GuidanceMC(flow, data_condition).to(device)
data_samples = sampler.sample(num_samples, guidance=data_guidance, num_steps=NUM_STEPS).detach()

fig, ax = plt.subplots(figsize=(6, 3.6))
price_ax = plot_paths(ax, data_samples, "Conditioned on data")
plot_observations(price_ax, observations, [0], obs_mask, transform=lambda x: 100.0 * torch.exp(x))
plt.tight_layout()

## 6. Conditioned on an Event

Now condition on the market shock alone: the price index drops below $87$ at some point in the year **and** volatility spikes above $0.42$. `MarketShock` is a `BinaryCondition` with one margin per component, using smooth (log-sum-exp) extremes over time; the price threshold is applied on the log scale, $\min_t X_t < \log 0.87$. Guidance must satisfy both at once.

In [ ]:
class MarketShock(BinaryCondition):
    """Minimum price index below `min_price` and maximum volatility above `max_vol`."""

    def __init__(self, min_price: float, max_vol: float, v: float = 1e-5, smooth_tau: float = 0.025):
        super().__init__(v=v)
        self.log_min_price = torch.log(torch.tensor(min_price / 100.0)).item()
        self.max_vol = max_vol
        self.smooth_tau = smooth_tau

    def smooth_eval_condition(self, x: torch.Tensor) -> torch.Tensor:
        log_price = x[:, 0, :]
        volatility = x[:, 1, :]
        components = torch.stack(
            [
                self.log_min_price - smooth_min(log_price, self.smooth_tau),
                smooth_max(volatility, self.smooth_tau) - self.max_vol,
            ],
            dim=-1,
        )
        return components[:, None, :]  # (B, 1, 2)


event_condition = MarketShock(MIN_PRICE, MAX_VOL)
event_guidance = GuidanceMC(flow, event_condition).to(device)
event_samples = sampler.sample(num_samples, guidance=event_guidance, num_steps=NUM_STEPS).detach()

event_prior_samples = flow.map(torch.randn(20_000, 2, num_times, device=device))  # large prior batch for the rate
print(f"P(market shock): prior={event_rate(event_condition, event_prior_samples):.3f}  "
      f"guided={event_rate(event_condition, event_samples):.3f}")

fig, ax = plt.subplots(figsize=(6, 3.6))
price_ax = plot_paths(ax, event_samples, "Conditioned on event")
show_event(ax, price_ax, loc="upper right")
plt.tight_layout()

## 7. Conditioned on Both

`ProductCondition` multiplies the two likelihoods, so guidance must follow the observed early prices **and** produce the shock later in the year.

In [ ]:
both_guidance = GuidanceMC(flow, ProductCondition([data_condition, event_condition])).to(device)
both_samples = sampler.sample(num_samples, guidance=both_guidance, num_steps=NUM_STEPS).detach()

print(f"P(market shock) given both: {event_rate(event_condition, both_samples):.3f}")
obs_error = (both_samples[:, 0, obs_mask] - truth[None, 0, obs_mask]).abs().mean().item()
print(f"mean |log-price error| at the observed points: {obs_error:.4f} (observation noise sd {OBS_NOISE})")

## 8. Summary

The four regimes side by side, as in Figure 4 of the LatentFlow paper. The same prior, flow and sampler are used throughout; only the condition changes.

In [ ]:
panels = [
    (prior_samples, "Prior"),
    (data_samples, "Conditioned on data"),
    (event_samples, "Conditioned on event"),
    (both_samples, "Conditioned on both"),
]
fig, axes = plt.subplots(1, 4, figsize=(17, 3.6), sharex=True)
for i, (ax, (samples, panel_title)) in enumerate(zip(axes, panels)):
    price_ax = plot_paths(ax, samples, panel_title, labels=False)
    if i == 0:
        ax.set_ylabel("volatility", color=COLORS["volatility"])
    if i == 3:
        price_ax.set_ylabel("price index", color=COLORS["price index"])
    if i in (1, 3):
        plot_observations(price_ax, observations, [0], obs_mask, transform=lambda x: 100.0 * torch.exp(x))
    if i in (2, 3):
        show_event(ax, price_ax, loc="upper right")
channel_legend(axes[0], loc="upper left")
fig.suptitle("Heston stochastic volatility", y=1.03, fontsize=14, fontweight="bold")
plt.tight_layout()